# **TUTORIAL** 5o: Run the optimization

Target setting starts from allocated carrying capacities and searches for
circular economy (CE) performance levels whose environmental burdens comply
with those carrying capacities. `run_optimization(...)` combines an initial
exploration of CE performance target levels with progressive local refinement.
The user selects a completed foreground scenario as the parent.

## Before starting...

Complete Tutorials 1--4 for that foreground and prepare the allocated carrying
capacity input described below. Monitoring follows
[Tutorial 5m](5m_run_asr.ipynb).


In [ ]:
from pathlib import Path

from pytoce import run_optimization, set_workspace

## Select the workspace

Run `set_workspace(...)` before any later function call. It sets the active
workspace root for the current Python session. Use the directory in which the
preceding tutorials were completed.


In [ ]:
workspace = Path.home() / "Documents" / "pytoce_workspace"
workspace_report = set_workspace(workspace)

## Add the allocated carrying capacity result

Calculate allocated carrying capacities following the
[optional pyaesa tutorial](../optional_workflows/pyaesa_allocated_carrying_capacity.ipynb),
or supply your own values based on your chosen allocation method and shares.
Store one complete result under
`data_raw/allocated_cc/results/<allocated_cc_result>/`, containing
`allocated_cc_result.yaml`, `annual_allocated_cc.csv`, and
`cumulative_gwp100_allocated_cc.csv`. Each table row records its source in
`cc_authority`.

The optional pyaesa tutorial uses the completed 2019 LCA results from Tutorial 4
and writes `pyaesa_plastics`, the result name used below. Its default cases are
`p5`, `p50`, and `p95`: the 5th, 50th (median), and 95th percentiles of the
allocated carrying capacity distributions. Each case is assessed separately.

The optional tutorial uses steady-state carrying capacities for annual assessment
and a dynamic climate change carrying capacity for cumulative GWP100 over
2019--2100.


In [ ]:
workspace_report.allocated_cc_results

## Run the three optimization blocks

This example uses *2019frozen_SSP2demand* through its scenario name
`frozen_2019_default_baseline`. Select another foreground name or a sequence
to start from other CE performance trajectories. Each foreground needs completed
MFA and baseline LCA results. Use the same reporting `lcia_method` selection
in the LCA and optimization calls.

Replace `pyaesa_plastics` if you supplied another allocated carrying capacity
result. `allocated_cc_case=None` selects every available case.

`background_scenario=None` selects the four REMIND SSP2 background scenarios
described in [Tutorial 2](2_prepare_backgrounds.ipynb). Pass a name or a
sequence, such as `("remind_ssp2_npi", "remind_ssp2_pkbudg650")`, to assess the
requested backgrounds. Baseline LCA results are required for each selected
background. Building the shared LCA responses requires all four prepared
backgrounds; completed responses are reused.

`optimization_lcia_method="ef31"` uses all 16 EF 3.1 impact categories and
cumulative GWP100 for constraints, ranking, and selection. `lcia_method` selects
the environmental burdens reported for each selected pathway: EF 3.1, PB-LCIA,
both groups, or one impact category identified by its full Brightway method
tuple.

The three blocks vary different CE performance families:

- `demand_lifetime_frontier`: Block I, demand reduction and lifetime extension.
- `circularity_feedstock_anchor`: Block II, collection, treatment composition,
  and virgin feedstock composition.
- `integrated_frontier`: Block III, all five CE performance families together.

CE performance families outside the selected block retain their parent
trajectories. Block III compares results from Blocks I and II with joint
combinations across all five families.

### Choose the search resolution

By default, pyToCE uses an initial exploration spacing of 20 percentage points,
followed by refinement stages at 10 and 5 percentage points. Use
`initial_grid_step_pct` and `endpoint_refinement_steps_pct` to select the
spacings. The initial spacing accepts 10, 20, 25, 50, or 100 percentage points.
Refinement spacings must be distinct positive integers, strictly decreasing
and below the initial spacing; `(5,)` requests a single stage.

At each refinement stage, Blocks I and II are evaluated before Block III so that
their updated results can be used in the joint search. Selection compares the
combinations evaluated under the chosen parameter bounds and refinement
spacings. The [workflow reference](https://github.com/Ike-EDB/pyToCE/blob/main/docs/workflow_reference.md#search-blocks-and-refinement)
describes the initial designs, combinations derived from earlier results, and
progressive refinement.

### Annual trajectories and demand-timing repair

For each tested combination, the fraction of change from the parent toward
the selected target levels increases linearly from zero in 2019 to one in 2050
and remains one thereafter. Demand reduction and lifetime extension are relative
to the corresponding parent values in each year. Collection, treatment
composition, and virgin feedstock composition reach common absolute target
shares while starting from regional parent values. The four treatment shares
sum to 100%, as do the three virgin feedstock shares.

When mean lifetime changes, the standard deviation changes proportionally to
retain the coefficient of variation of the original lifetime distributions.

The dynamic MFA first checks the annual stocks and material flows through
2100. If a linear trajectory violates the MFA constraints and demand reduction
is positive, demand-timing repair adjusts the annual timing of demand reduction
before 2050. The target levels, completion year, and other CE performance
trajectories remain fixed. A linear program minimizes the average absolute
difference from the initial demand-reduction trajectory. If needed, additional
annual timing changes are tested against the complete MFA. Trajectories
satisfying the MFA constraints proceed to environmental assessment. The
[workflow reference](https://github.com/Ike-EDB/pyToCE/blob/main/docs/workflow_reference.md#mfa-feasibility-and-demand-timing-repair)
describes the stock constraints and repair procedure.

### Execution and reuse

`candidate_workers` selects the maximum number of concurrent candidate
evaluation processes. It accepts integers from 1 through 61 and defaults to 1.
Choose a count that fits the available CPU cores and memory; each process
retains its own inputs and MFA state.

Matching completed candidate calculations are reused, including when a search
resumes after an interruption. Processing times reported for each worker can
overlap when workers run concurrently.


In [ ]:
optimization_report = run_optimization(
    foreground_scenario="frozen_2019_default_baseline",
    allocated_cc_result="pyaesa_plastics",
    background_scenario=None,
    candidate_workers=1,
    optimization_lcia_method="ef31",
    lcia_method=("ef31", "pb_lcia"),
    optimization_block=(
        "demand_lifetime_frontier",
        "circularity_feedstock_anchor",
        "integrated_frontier",
    ),
    allocated_cc_case=None,
    initial_grid_step_pct=20,
    endpoint_refinement_steps_pct=(10, 5),
)
optimization_report.as_dict()

## Read the selected interventions

`optimization_report.products` gives the result paths for each foreground.
Read `selected_coordinates.parquet` for the selected CE performance target
levels and the MFA validation and demand-repair outcomes. Use
`candidate_identity` to link these results with
`selected_annual_interventions.parquet`, which reports annual CE performance
changes as fractions, including repaired demand-reduction trajectories. The
guide at `results/<foreground_scenario>/optimization_results/README.md`
defines the tables, column names, units, and result statuses.

Compliance requires an Absolute Sustainability Ratio (ASR) at most 1 for all
16 EF 3.1 impact categories in each of the 12 regions for every year from 2020
through 2100, and for cumulative GWP100 in each region over 2019--2100.

The unchanged parent is retained when it meets every constraint. Among
compliant combinations, selection first minimizes intervention distance, with
equal weight given to the CE performance families considered in the block.
Remaining ties favor the combination whose highest ASR is closest to one.

If none of the evaluated combinations complies, fallback combinations are
ranked successively by the numbers of fully compliant regions, fully compliant
impact categories, compliant region-impact category pairs, and individual
compliant ASRs. Remaining ties are resolved by minimizing maximum positive ASR
exceedance, mean positive ASR exceedance, and intervention distance, in that
order. The [selection rules](https://github.com/Ike-EDB/pyToCE/blob/main/docs/workflow_reference.md#selection-rules)
define these quantities.

The retained target levels are also evaluated with completion in 2100 to
assess the effect of the transition rate. Each foreground, background,
allocated carrying capacity case, and search block retains its own results.


In [ ]:
[product.as_dict() for product in optimization_report.products]

## Reuse or extend a completed search

Repeating the same call reuses the completed result. To add refinement at one
percentage point after this run finishes, change
`endpoint_refinement_steps_pct` to `(10, 5, 1)` and run it again. Keep
`initial_grid_step_pct=20` and the same scientific inputs, backgrounds, blocks,
and allocated carrying capacity cases.

With the default `refresh=False`, compatible completed stages are reused and
the additional refinement stage is calculated. Updated Block I and II results
provide starting combinations for Block III, and matching candidate calculations
are reused. Each refinement schedule has its own selected results; use the paths
in `optimization_report.products` to locate them.

Use `refresh=True` to rewrite the selected results from completed prerequisites
and matching candidate calculations. The
[workflow reference](https://github.com/Ike-EDB/pyToCE/blob/main/docs/workflow_reference.md#search-blocks-and-refinement)
gives the conditions for reusing and extending a completed search.
